In [1]:
import pandas as pd
import duckdb

df_device_inspection = pd.DataFrame({
    "inspection_id": [
        101, 102, 103, 104,
        201, 202, 203,
        301, 302, 303, 304
    ],
    "device_id": [
        "R05", "R05", "R05", "R05",
        "R16", "R16", "R16",
        "R34", "R34", "R34", "R34"
    ],
    "inspection_time": [
        "2026-07-28 08:00:00",
        "2026-07-28 08:40:00",
        "2026-07-28 10:00:00",
        "2026-07-28 11:30:00",

        "2026-07-28 09:00:00",
        "2026-07-28 10:30:00",
        "2026-07-28 13:00:00",

        "2026-07-28 07:30:00",
        "2026-07-28 08:00:00",
        "2026-07-28 09:15:00",
        "2026-07-28 09:45:00"
    ],
    "inspection_result": [
        "OK", "OK", "WARNING", "OK",
        "OK", "ERROR", "OK",
        "OK", "WARNING", "WARNING", "OK"
    ]
})

df_device_inspection["inspection_time"] = pd.to_datetime(
    df_device_inspection["inspection_time"]
)

df_device_inspection

,inspection_id,device_id,inspection_time,inspection_result
0,101,R05,2026-07-28 08:00:00,OK
1,102,R05,2026-07-28 08:40:00,OK
2,103,R05,2026-07-28 10:00:00,WARNING
3,104,R05,2026-07-28 11:30:00,OK
4,201,R16,2026-07-28 09:00:00,OK
5,202,R16,2026-07-28 10:30:00,ERROR
6,203,R16,2026-07-28 13:00:00,OK
7,301,R34,2026-07-28 07:30:00,OK
8,302,R34,2026-07-28 08:00:00,WARNING
9,303,R34,2026-07-28 09:15:00,WARNING


# SQL Daily Review：计算两次巡检时间间隔

## 题目背景

设备会按照时间进行巡检，每条记录表示一次巡检。

现在需要计算每台设备每次巡检距离上一次巡检经过了多少分钟。

---

## 题目要求

对于每条巡检记录，计算：

1. 上一次巡检时间；
2. 与上一次巡检相隔的分钟数。

第一条巡检由于没有上一条记录：

```text
previous_inspection_time = NULL
interval_minutes = NULL
```

---

### 输出字段

| 字段 | 含义 |
|------|------|
| device_id | 设备编号 |
| inspection_id | 巡检编号 |
| inspection_time | 当前巡检时间 |
| previous_inspection_time | 上一次巡检时间 |
| interval_minutes | 两次巡检间隔（分钟） |

---

### 计算规则

每台设备分别计算。

按照：

```text
inspection_time ASC
inspection_id ASC
```

确定巡检顺序。

---

### 最终排序

按照：

1. device_id ASC
2. inspection_time ASC
3. inspection_id ASC

---

## 解题要求

- 使用 `LAG()`
- 使用 CTE
- 不使用自连接
- 使用 DuckDB 的

```sql
DATE_DIFF('minute', previous_inspection_time, inspection_time)
```

计算分钟差。

In [2]:
query = '''

WITH previous_time_table AS (
    SELECT
        inspection_id,
        device_id,
        inspection_time,
        LAG(inspection_time)
        OVER(
            PARTITION BY device_id ORDER BY inspection_time,inspection_id
        ) AS previous_inspection_time
    FROM df_device_inspection
)
SELECT
    device_id,
    inspection_id,
    inspection_time,
    previous_inspection_time,
    DATE_DIFF('minute',previous_inspection_time,inspection_time) AS interval_minutes
FROM previous_time_table
ORDER BY device_id,inspection_time,inspection_id
'''
df = duckdb.execute(query).fetchdf()
df

,device_id,inspection_id,inspection_time,previous_inspection_time,interval_minutes
0,R05,101,2026-07-28 08:00:00,NaT,<NA>
1,R05,102,2026-07-28 08:40:00,2026-07-28 08:00:00,40
2,R05,103,2026-07-28 10:00:00,2026-07-28 08:40:00,80
3,R05,104,2026-07-28 11:30:00,2026-07-28 10:00:00,90
4,R16,201,2026-07-28 09:00:00,NaT,<NA>
5,R16,202,2026-07-28 10:30:00,2026-07-28 09:00:00,90
6,R16,203,2026-07-28 13:00:00,2026-07-28 10:30:00,150
7,R34,301,2026-07-28 07:30:00,NaT,<NA>
8,R34,302,2026-07-28 08:00:00,2026-07-28 07:30:00,30
9,R34,303,2026-07-28 09:15:00,2026-07-28 08:00:00,75
